# ĐỒ ÁN MÔN HỌC: XỬ LÝ ẢNH — NHẬN DIỆN CHẤT LƯỢNG GẠO
## Phân hệ 4: Phân loại chất lượng & Thống kê số liệu — Thắng Card lord

Notebook này phân tích đặc tính từ tập ảnh mẫu có nhãn để thiết lập ngưỡng tham chiếu, áp dụng quy tắc phân loại chất lượng và màu sắc cho từng hạt gạo từ bảng đặc trưng của Thành viên 3, đồng thời tổng hợp thống kê và trực quan hóa kết quả[cite: 4, 5, 7].

**Đặc tả kỹ thuật:**
* **Phân tích mẫu & quy tắc phân loại (`if - else`):** Phân tích phạm vi giá trị của các nhóm mẫu (`Tốt`, `Vỡ`, `Kém`, `Tệ`, `Tạp chất`...), xác định ngưỡng kích thước/hình dạng và ngưỡng màu (`Bình thường`, `Vàng`, `Sẫm`), phân loại bằng cấu trúc `if - else` và xử lý rõ ràng trường hợp `"Không xác định"`[cite: 5, 7].
* **Thống kê số lượng & tỷ lệ:** Đếm tổng số hạt, số lượng theo từng nhãn chất lượng và nhãn màu sắc, tính tỷ lệ phần trăm `(count / total) * 100` cho từng nhóm[cite: 7].
* **Thống kê kích thước & phân bố màu:** Tính kích thước hạt gạo trung bình (`np.mean()`), phân tích thống kê `HSV` và vẽ biểu đồ trực quan bằng `Matplotlib`[cite: 7].

In [ ]:
# %% [code]
import os
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ==============================================================================
# 1. CẤU HÌNH ĐƯỜNG DẪN TƯƠNG ĐỐI
# ==============================================================================
try:
    PROJECT_ROOT = os.path.dirname(os.path.abspath(__file__))
except NameError:
    PROJECT_ROOT = os.getcwd()

REF_COLOR_DIR = os.path.join(
    PROJECT_ROOT, "processed_dataset", "1_reference_labeled", "color_clean"
)
REF_CSV_PATH = os.path.join(
    PROJECT_ROOT, "processed_dataset", "1_reference_labeled", "reference_labels.csv"
)

TV3_FEATURES_CLEAN_CSV = os.path.join(
    PROJECT_ROOT, "features_dataset", "grain_features_clean.csv"
)
TV3_FEATURES_ALL_CSV = os.path.join(
    PROJECT_ROOT, "features_dataset", "grain_features.csv"
)

CLS_OUTPUT_DIR = os.path.join(PROJECT_ROOT, "classification_results")
CLASSIFIED_GRAINS_CSV = os.path.join(CLS_OUTPUT_DIR, "classified_grains.csv")
SUMMARY_STATS_CSV = os.path.join(CLS_OUTPUT_DIR, "quality_color_summary.csv")
THRESHOLDS_REPORT_CSV = os.path.join(CLS_OUTPUT_DIR, "learned_reference_stats.csv")

REF_FOLDER_TO_LABEL = {
    "Tot": "Tốt",
    "Vo": "Vỡ",
    "Kem": "Kém",
    "Te": "Tệ",
    "Tap_chat": "Tạp chất",
}

SUPPORTED_EXTS = (".jpg", ".jpeg", ".png", ".bmp")

# Ngưỡng HSV trích xuất pixel vàng / sẫm (đồng bộ với TV3)
HSV_PIXEL_CFG = {
    "yellow_h_min": 15,
    "yellow_h_max": 38,
    "yellow_s_min": 40,
    "yellow_v_min": 60,
    "dark_v_max": 85,
    "dark_s_min": 30,
}

print(f"Thư mục gốc: {PROJECT_ROOT}")

In [ ]:
# %% [code]
def segment_reference_object(bgr_img: np.ndarray) -> np.ndarray:
    """
    Tách vùng vật thể chính trên ảnh mẫu (Tốt, Vỡ, Kém, Tệ, Tạp chất) bằng xử lý ảnh.
    """
    gray = cv2.cvtColor(bgr_img, cv2.COLOR_BGR2GRAY)
    blur = cv2.GaussianBlur(gray, (5, 5), 0)
    clahe = cv2.createCLAHE(clipLimit=2.5, tileGridSize=(8, 8))
    enh = clahe.apply(blur)

    _, mask = cv2.threshold(enh, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    if np.count_nonzero(mask) > 0.65 * mask.size:
        mask = cv2.bitwise_not(mask)

    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel, iterations=1)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel, iterations=1)

    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        return np.ones_like(gray, dtype=np.uint8) * 255

    main_cnt = max(contours, key=cv2.contourArea)
    if cv2.contourArea(main_cnt) < 20:
        return np.ones_like(gray, dtype=np.uint8) * 255

    clean_mask = np.zeros_like(gray, dtype=np.uint8)
    cv2.drawContours(clean_mask, [main_cnt], -1, 255, thickness=-1)
    return clean_mask


def extract_reference_sample_features(bgr_img: np.ndarray) -> dict:
    """
    Đo cùng bộ đặc trưng hình dạng và màu sắc như TV3 trên ảnh mẫu có nhãn.
    """
    mask = segment_reference_object(bgr_img)
    mask_bool = mask > 0
    num_px = max(1, int(np.count_nonzero(mask_bool)))

    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if contours:
        cnt = max(contours, key=cv2.contourArea)
        area = float(cv2.contourArea(cnt))
        perimeter = float(cv2.arcLength(cnt, True))
        (_, _), (rw, rh), _ = cv2.minAreaRect(cnt)
        length = float(max(rw, rh))
        width = float(min(rw, rh))
        aspect_ratio = float(length / width) if width > 1e-6 else 1.0
        circularity = float((4.0 * np.pi * area) / (perimeter * perimeter)) if perimeter > 1e-6 else 0.0
        hull_area = float(cv2.contourArea(cv2.convexHull(cnt)))
        solidity = float(area / hull_area) if hull_area > 1e-6 else 0.0
    else:
        area = float(num_px)
        length = float(np.sqrt(num_px))
        width = length
        aspect_ratio = 1.0
        circularity = 0.0
        solidity = 1.0

    hsv = cv2.cvtColor(bgr_img, cv2.COLOR_BGR2HSV)
    h_vals = hsv[:, :, 0][mask_bool].astype(np.float64)
    s_vals = hsv[:, :, 1][mask_bool].astype(np.float64)
    v_vals = hsv[:, :, 2][mask_bool].astype(np.float64)

    yellow_cond = (
        (h_vals >= HSV_PIXEL_CFG["yellow_h_min"])
        & (h_vals <= HSV_PIXEL_CFG["yellow_h_max"])
        & (s_vals >= HSV_PIXEL_CFG["yellow_s_min"])
        & (v_vals >= HSV_PIXEL_CFG["yellow_v_min"])
    )
    dark_cond = (v_vals <= HSV_PIXEL_CFG["dark_v_max"]) | (
        (v_vals <= 110) & (s_vals >= HSV_PIXEL_CFG["dark_s_min"])
    )

    return {
        "shape_area": area,
        "shape_length": length,
        "shape_width": width,
        "shape_aspect_ratio": aspect_ratio,
        "shape_circularity": circularity,
        "shape_solidity": solidity,
        "color_h_mean": float(np.mean(h_vals)),
        "color_s_mean": float(np.mean(s_vals)),
        "color_v_mean": float(np.mean(v_vals)),
        "color_yellow_ratio": float(np.count_nonzero(yellow_cond)) / num_px,
        "color_dark_ratio": float(np.count_nonzero(dark_cond)) / num_px,
    }


def load_reference_records() -> list[dict]:
    records = []
    if os.path.exists(REF_CSV_PATH):
        df_ref = pd.read_csv(REF_CSV_PATH)
        for _, row in df_ref.iterrows():
            c_path = str(row.get("color_path", "")).strip()
            if c_path and not os.path.isabs(c_path):
                c_path = os.path.join(PROJECT_ROOT, c_path)
            if not c_path or not os.path.exists(c_path):
                continue
            lbl = str(row.get("label_name", "")).strip()
            if not lbl:
                lbl = REF_FOLDER_TO_LABEL.get(str(row.get("folder_code", "")).strip(), "")
            if lbl:
                records.append({"label_name": lbl, "img_path": c_path})

    if not records and os.path.exists(REF_COLOR_DIR):
        for fcode, lbl in REF_FOLDER_TO_LABEL.items():
            sdir = os.path.join(REF_COLOR_DIR, fcode)
            if not os.path.exists(sdir):
                continue
            for fn in sorted(os.listdir(sdir)):
                if fn.lower().endswith(SUPPORTED_EXTS):
                    records.append({"label_name": lbl, "img_path": os.path.join(sdir, fn)})
    return records


def build_reference_profiles_and_thresholds() -> tuple[dict, pd.DataFrame]:
    """
    Phân tích phạm vi giá trị từng nhóm chất lượng trên dataset mẫu để rút ra:
    1. Ngưỡng phân loại màu (Bình thường, Vàng, Sẫm) bám theo mẫu Tốt vs mẫu lỗi.
    2. Hồ sơ đặc trưng chuẩn hóa (màu + tỷ lệ hình dạng) của từng nhóm chất lượng trong mẫu.
    """
    ref_records = load_reference_records()
    rows = []
    for rec in ref_records:
        img = cv2.imread(rec["img_path"])
        if img is None:
            continue
        feat = extract_reference_sample_features(img)
        feat["label_name"] = rec["label_name"]
        rows.append(feat)

    if not rows:
        raise FileNotFoundError("Không tìm thấy dữ liệu ảnh mẫu trong 'processed_dataset/1_reference_labeled'.")

    df_ref = pd.DataFrame(rows)

    # Tính kích thước chuẩn của hạt Tốt trong dataset mẫu để quy đổi tỷ lệ tương đối
    good_df = df_ref[df_ref["label_name"] == "Tốt"]
    ref_base_area = float(good_df["shape_area"].median()) if not good_df.empty else float(df_ref["shape_area"].median())
    ref_base_length = float(good_df["shape_length"].median()) if not good_df.empty else float(df_ref["shape_length"].median())

    df_ref["rel_area"] = df_ref["shape_area"] / max(1e-6, ref_base_area)
    df_ref["rel_length"] = df_ref["shape_length"] / max(1e-6, ref_base_length)

    # Xây dựng bảng thống kê phạm vi giá trị từng nhóm mẫu
    cols_to_summarize = [
        "shape_area", "shape_length", "shape_aspect_ratio", "shape_solidity",
        "rel_area", "rel_length",
        "color_h_mean", "color_s_mean", "color_v_mean",
        "color_yellow_ratio", "color_dark_ratio",
    ]
    summary_rows = []
    class_profiles = {}

    for lbl, grp in df_ref.groupby("label_name"):
        row_stat = {"label_name": lbl, "sample_count": len(grp)}
        for c in cols_to_summarize:
            row_stat[f"{c}_mean"] = round(float(grp[c].mean()), 4)
            row_stat[f"{c}_min"] = round(float(grp[c].min()), 4)
            row_stat[f"{c}_max"] = round(float(grp[c].max()), 4)
        summary_rows.append(row_stat)

        # Lưu hồ sơ trung vị & độ trải của từng nhóm mẫu
        class_profiles[lbl] = {
            c: {
                "median": float(grp[c].median()),
                "q25": float(grp[c].quantile(0.25)),
                "q75": float(grp[c].quantile(0.75)),
                "iqr": max(1e-4, float(grp[c].quantile(0.75) - grp[c].quantile(0.25))),
            }
            for c in cols_to_summarize
        }

    df_ref_summary = pd.DataFrame(summary_rows)

    # Suy ra ngưỡng màu bình thường / vàng / sẫm từ phân phối của nhóm Tốt so với Kém/Tệ/Tạp chất
    if not good_df.empty:
        good_y_q75 = float(good_df["color_yellow_ratio"].quantile(0.75))
        good_d_q75 = float(good_df["color_dark_ratio"].quantile(0.75))
        good_v_q25 = float(good_df["color_v_mean"].quantile(0.25))
    else:
        good_y_q75, good_d_q75, good_v_q25 = 0.15, 0.15, 110.0

    defect_df = df_ref[df_ref["label_name"].isin(["Kém", "Tệ", "Tạp chất"])]
    if not defect_df.empty:
        def_y_med = float(defect_df["color_yellow_ratio"].median())
        def_d_med = float(defect_df["color_dark_ratio"].median())
        yellow_thresh = float(np.clip((good_y_q75 + def_y_med) / 2.0, good_y_q75 + 0.02, 0.65))
        dark_thresh = float(np.clip((good_d_q75 + def_d_med) / 2.0, good_d_q75 + 0.02, 0.65))
    else:
        yellow_thresh = min(0.55, good_y_q75 + 0.08)
        dark_thresh = min(0.55, good_d_q75 + 0.08)

    # Ngưỡng hạt vỡ so với hạt Tốt từ mẫu
    broken_df = df_ref[df_ref["label_name"] == "Vỡ"]
    if not broken_df.empty:
        broken_rel_area_thresh = float(np.clip((broken_df["rel_area"].quantile(0.75) + 1.0) / 2.0, 0.45, 0.85))
        broken_rel_len_thresh = float(np.clip((broken_df["rel_length"].quantile(0.75) + 1.0) / 2.0, 0.50, 0.88))
    else:
        broken_rel_area_thresh = 0.68
        broken_rel_len_thresh = 0.74

    thresholds = {
        "yellow_ratio_thresh": round(yellow_thresh, 4),
        "dark_ratio_thresh": round(dark_thresh, 4),
        "dark_v_mean_thresh": round(max(60.0, good_v_q25 * 0.82), 2),
        "broken_rel_area_thresh": round(broken_rel_area_thresh, 4),
        "broken_rel_len_thresh": round(broken_rel_len_thresh, 4),
        "class_profiles": class_profiles,
    }

    return thresholds, df_ref_summary


REF_MODEL, df_ref_stats = build_reference_profiles_and_thresholds()
print("Ngưỡng học được từ dataset mẫu:")
for k, v in REF_MODEL.items():
    if k != "class_profiles":
        print(f"  - {k}: {v}")
df_ref_stats

In [12]:
# %% [code]
def classify_grain_color_by_ref(row: pd.Series, ref_model: dict) -> str:
    """
    Phân loại màu sắc hạt dựa trên ngưỡng học từ dataset mẫu:
    - 'Bình thường'
    - 'Vàng'
    - 'Sẫm'
    - 'Không xác định'
    """
    h_mean = row.get("color_h_mean", np.nan)
    s_mean = row.get("color_s_mean", np.nan)
    v_mean = row.get("color_v_mean", np.nan)
    y_ratio = row.get("color_yellow_ratio", np.nan)
    d_ratio = row.get("color_dark_ratio", np.nan)

    if any(pd.isna(x) or np.isinf(x) for x in [h_mean, s_mean, v_mean, y_ratio, d_ratio]):
        return "Không xác định"

    y_thresh = ref_model["yellow_ratio_thresh"]
    d_thresh = ref_model["dark_ratio_thresh"]
    v_dark = ref_model["dark_v_mean_thresh"]

    is_yellow = (y_ratio >= y_thresh) or ((15.0 <= h_mean <= 38.0) and (s_mean >= 50.0) and (y_ratio >= y_thresh * 0.75))
    is_dark = (d_ratio >= d_thresh) or (v_mean <= v_dark and d_ratio >= d_thresh * 0.65)

    if is_yellow and is_dark:
        if y_ratio >= d_ratio:
            return "Vàng"
        else:
            return "Sẫm"
    elif is_yellow:
        return "Vàng"
    elif is_dark:
        return "Sẫm"
    elif 0.0 <= v_mean <= 255.0 and 0.0 <= s_mean <= 255.0:
        return "Bình thường"
    else:
        return "Không xác định"


def compute_similarity_distance_to_profile(sample_vals: dict, profile: dict) -> float:
    """
    Tính khoảng cách chuẩn hóa giữa đặc trưng hạt (màu sắc + kích thước tương đối + hình dạng)
    với hồ sơ của 1 nhóm trong dataset mẫu.
    """
    weights = {
        "rel_area": 1.6,
        "rel_length": 1.6,
        "shape_aspect_ratio": 1.2,
        "shape_solidity": 1.2,
        "color_h_mean": 1.0,
        "color_s_mean": 1.2,
        "color_v_mean": 1.3,
        "color_yellow_ratio": 1.8,
        "color_dark_ratio": 1.8,
    }
    total_dist = 0.0
    total_w = 0.0

    for feat_name, w in weights.items():
        if feat_name not in sample_vals or feat_name not in profile:
            continue
        val = sample_vals[feat_name]
        if pd.isna(val) or np.isinf(val):
            continue
        med = profile[feat_name]["median"]
        scale = max(profile[feat_name]["iqr"], abs(med) * 0.15, 0.05)
        total_dist += w * abs(val - med) / scale
        total_w += w

    return (total_dist / total_w) if total_w > 0 else float("inf")


def classify_grain_quality_by_ref(
    row: pd.Series,
    batch_base_area: float,
    batch_base_length: float,
    ref_model: dict,
) -> str:
    """
    Phân loại chất lượng hạt bằng cấu trúc if - else kết hợp ngưỡng & độ tương đồng
    với từng nhóm trong dataset mẫu ('Tốt', 'Vỡ', 'Kém', 'Tệ', 'Tạp chất', 'Không xác định').
    """
    area = row.get("shape_area", np.nan)
    length = row.get("shape_length", np.nan)
    aspect_ratio = row.get("shape_aspect_ratio", np.nan)
    solidity = row.get("shape_solidity", np.nan)
    h_mean = row.get("color_h_mean", np.nan)
    s_mean = row.get("color_s_mean", np.nan)
    v_mean = row.get("color_v_mean", np.nan)
    y_ratio = row.get("color_yellow_ratio", np.nan)
    d_ratio = row.get("color_dark_ratio", np.nan)
    color_label = row.get("color_label", "Không xác định")

    # 1. Xử lý trường hợp thiếu / bất thường / không xác định
    if any(pd.isna(x) or np.isinf(x) for x in [area, length, aspect_ratio, solidity, v_mean, y_ratio, d_ratio]):
        return "Không xác định"
    if area <= 0 or length <= 0 or color_label == "Không xác định":
        return "Không xác định"

    # Quy đổi kích thước tương đối so với hạt nguyên chuẩn của mẻ gạo (tương đương rel_area, rel_length của mẫu)
    rel_area = float(area) / max(1e-6, batch_base_area)
    rel_length = float(length) / max(1e-6, batch_base_length)

    profiles = ref_model.get("class_profiles", {})
    broken_area_th = ref_model["broken_rel_area_thresh"]
    broken_len_th = ref_model["broken_rel_len_thresh"]

    # 2. Kiểm tra điều kiện rõ rệt bằng if - else trước
    if (aspect_ratio < 1.25 and rel_area > 1.75) or (solidity < 0.62):
        if "Tạp chất" in profiles:
            return "Tạp chất"
        else:
            return "Tệ"
    elif (rel_area < broken_area_th) and (rel_length < broken_len_th):
        return "Vỡ"
    elif (color_label == "Bình thường") and (rel_area >= broken_area_th) and (solidity >= 0.80):
        # Kiểm tra độ gần với mẫu Tốt so với Kém/Tệ
        sample_dict = {
            "rel_area": rel_area,
            "rel_length": rel_length,
            "shape_aspect_ratio": float(aspect_ratio),
            "shape_solidity": float(solidity),
            "color_h_mean": float(h_mean),
            "color_s_mean": float(s_mean),
            "color_v_mean": float(v_mean),
            "color_yellow_ratio": float(y_ratio),
            "color_dark_ratio": float(d_ratio),
        }
        if "Tốt" in profiles:
            dist_good = compute_similarity_distance_to_profile(sample_dict, profiles["Tốt"])
            if dist_good <= 2.5:
                return "Tốt"
            else:
                return "Kém"
        else:
            return "Tốt"
    else:
        # 3. Các hạt nằm vùng giao thoa: so sánh tổng hợp (màu sắc + kích thước + hình dạng) với từng nhóm mẫu
        sample_dict = {
            "rel_area": rel_area,
            "rel_length": rel_length,
            "shape_aspect_ratio": float(aspect_ratio),
            "shape_solidity": float(solidity),
            "color_h_mean": float(h_mean),
            "color_s_mean": float(s_mean),
            "color_v_mean": float(v_mean),
            "color_yellow_ratio": float(y_ratio),
            "color_dark_ratio": float(d_ratio),
        }
        if not profiles:
            return "Không xác định"

        best_label = "Không xác định"
        best_dist = float("inf")
        for lbl, prof in profiles.items():
            d = compute_similarity_distance_to_profile(sample_dict, prof)
            if d < best_dist:
                best_dist = d
                best_label = lbl

        if best_dist > 4.0:
            return "Không xác định"
        elif best_label in ("Tốt", "Vỡ", "Kém", "Tệ", "Tạp chất"):
            return best_label
        else:
            return "Không xác định"

In [ ]:
# %% [code]
def load_tv3_features_df() -> pd.DataFrame:
    if os.path.exists(TV3_FEATURES_CLEAN_CSV):
        return pd.read_csv(TV3_FEATURES_CLEAN_CSV)
    if os.path.exists(TV3_FEATURES_ALL_CSV):
        return pd.read_csv(TV3_FEATURES_ALL_CSV)
    raise FileNotFoundError("Không tìm thấy file CSV đặc trưng của TV3.")


def run_tv4_pipeline(ref_model: dict = REF_MODEL) -> tuple[pd.DataFrame, pd.DataFrame, dict]:
    df = load_tv3_features_df().copy()
    os.makedirs(CLS_OUTPUT_DIR, exist_ok=True)

    # Mốc kích thước hạt nguyên chuẩn của mẻ gạo (lấy phân vị 65% để không bị kéo thấp bởi hạt vỡ)
    valid_areas = df["shape_area"].dropna()
    valid_lengths = df["shape_length"].dropna()
    batch_base_area = float(valid_areas.quantile(0.65)) if len(valid_areas) > 0 else 1.0
    batch_base_length = float(valid_lengths.quantile(0.65)) if len(valid_lengths) > 0 else 1.0

    # Phân loại màu sắc & chất lượng
    df["color_label"] = df.apply(lambda r: classify_grain_color_by_ref(r, ref_model), axis=1)
    df["quality_label"] = df.apply(
        lambda r: classify_grain_quality_by_ref(r, batch_base_area, batch_base_length, ref_model),
        axis=1,
    )

    total_grains = len(df)
    avg_area = float(np.mean(df["shape_area"])) if total_grains > 0 else 0.0
    avg_length = float(np.mean(df["shape_length"])) if total_grains > 0 else 0.0
    avg_width = float(np.mean(df["shape_width"])) if total_grains > 0 else 0.0

    quality_order = ["Tốt", "Vỡ", "Kém", "Tệ", "Tạp chất", "Không xác định"]
    color_order = ["Bình thường", "Vàng", "Sẫm", "Không xác định"]

    q_counts = df["quality_label"].value_counts().to_dict()
    c_counts = df["color_label"].value_counts().to_dict()

    summary_rows = []
    for lbl in quality_order:
        cnt = int(q_counts.get(lbl, 0))
        pct = float((cnt / total_grains) * 100.0) if total_grains > 0 else 0.0
        sub = df[df["quality_label"] == lbl]
        summary_rows.append({
            "group_type": "Chất lượng",
            "label": lbl,
            "count": cnt,
            "percentage": round(pct, 2),
            "mean_area_px": round(float(np.mean(sub["shape_area"])), 2) if len(sub) > 0 else np.nan,
            "mean_length_px": round(float(np.mean(sub["shape_length"])), 2) if len(sub) > 0 else np.nan,
        })

    for lbl in color_order:
        cnt = int(c_counts.get(lbl, 0))
        pct = float((cnt / total_grains) * 100.0) if total_grains > 0 else 0.0
        sub = df[df["color_label"] == lbl]
        summary_rows.append({
            "group_type": "Màu sắc",
            "label": lbl,
            "count": cnt,
            "percentage": round(pct, 2),
            "mean_area_px": round(float(np.mean(sub["shape_area"])), 2) if len(sub) > 0 else np.nan,
            "mean_length_px": round(float(np.mean(sub["shape_length"])), 2) if len(sub) > 0 else np.nan,
        })

    df_summary = pd.DataFrame(summary_rows)

    hsv_stats = {
        "H_mean": round(float(np.mean(df["color_h_mean"])), 2) if total_grains > 0 else np.nan,
        "H_std": round(float(np.std(df["color_h_mean"])), 2) if total_grains > 0 else np.nan,
        "S_mean": round(float(np.mean(df["color_s_mean"])), 2) if total_grains > 0 else np.nan,
        "S_std": round(float(np.std(df["color_s_mean"])), 2) if total_grains > 0 else np.nan,
        "V_mean": round(float(np.mean(df["color_v_mean"])), 2) if total_grains > 0 else np.nan,
        "V_std": round(float(np.std(df["color_v_mean"])), 2) if total_grains > 0 else np.nan,
    }

    report = {
        "total_grains": int(total_grains),
        "avg_area_px": round(avg_area, 2),
        "avg_length_px": round(avg_length, 2),
        "avg_width_px": round(avg_width, 2),
        "quality_counts": {k: int(q_counts.get(k, 0)) for k in quality_order},
        "color_counts": {k: int(c_counts.get(k, 0)) for k in color_order},
        "hsv_stats": hsv_stats,
    }

    df.to_csv(CLASSIFIED_GRAINS_CSV, index=False, encoding="utf-8-sig")
    df_summary.to_csv(SUMMARY_STATS_CSV, index=False, encoding="utf-8-sig")
    df_ref_stats.to_csv(THRESHOLDS_REPORT_CSV, index=False, encoding="utf-8-sig")

    print("=" * 65)
    print("THỐNG KÊ CHẤT LƯỢNG & MÀU SẮC (ĐỐI CHIẾU DATASET MẪU)")
    print("=" * 65)
    print(f"Tổng số hạt              : {report['total_grains']}")
    print(f"Diện tích trung bình (px): {report['avg_area_px']}")
    print(f"Chiều dài trung bình (px): {report['avg_length_px']}")
    print(f"Chiều rộng trung bình(px): {report['avg_width_px']}")
    print("-" * 65)
    print("Phân bố chất lượng:")
    for lbl in quality_order:
        c = report["quality_counts"][lbl]
        p = (c / total_grains * 100.0) if total_grains > 0 else 0.0
        print(f"  - {lbl:<15}: {c:>5} hạt ({p:6.2f}%)")
    print("-" * 65)
    print("Phân bố màu sắc:")
    for lbl in color_order:
        c = report["color_counts"][lbl]
        p = (c / total_grains * 100.0) if total_grains > 0 else 0.0
        print(f"  - {lbl:<15}: {c:>5} hạt ({p:6.2f}%)")
    print("-" * 65)
    print("Thống kê HSV:", hsv_stats)
    print("=" * 65)

    return df, df_summary, report


df_classified, df_summary_stats, stats_report = run_tv4_pipeline()
df_summary_stats

In [ ]:
# %% [code]
def plot_tv4_statistics(df: pd.DataFrame, report: dict) -> None:
    if df.empty:
        print("Không có dữ liệu để vẽ biểu đồ.")
        return

    quality_labels = ["Tốt", "Vỡ", "Kém", "Tệ", "Tạp chất", "Không xác định"]
    q_counts = [report["quality_counts"].get(k, 0) for k in quality_labels]

    color_labels = ["Bình thường", "Vàng", "Sẫm", "Không xác định"]
    c_counts = [report["color_counts"].get(k, 0) for k in color_labels]

    fig, axes = plt.subplots(2, 2, figsize=(14, 10))

    axes[0, 0].bar(quality_labels, q_counts, color=["#2E8B57", "#E69F00", "#56B4E9", "#D55E00", "#7F7F7F", "#BDBDBD"])
    axes[0, 0].set_title("Số lượng hạt theo nhóm chất lượng")
    axes[0, 0].set_ylabel("Số lượng hạt")
    axes[0, 0].tick_params(axis="x", rotation=20)
    for idx, val in enumerate(q_counts):
        axes[0, 0].text(idx, val + max(1, max(q_counts) * 0.01), str(val), ha="center", va="bottom", fontsize=9)

    axes[0, 1].bar(color_labels, c_counts, color=["#4CAF50", "#FBC02D", "#5D4037", "#9E9E9E"])
    axes[0, 1].set_title("Số lượng hạt theo nhãn màu sắc")
    axes[0, 1].set_ylabel("Số lượng hạt")
    axes[0, 1].tick_params(axis="x", rotation=15)
    for idx, val in enumerate(c_counts):
        axes[0, 1].text(idx, val + max(1, max(c_counts) * 0.01), str(val), ha="center", va="bottom", fontsize=9)

    axes[1, 0].hist(df["color_v_mean"].dropna(), bins=25, alpha=0.75, label="Độ sáng (V)", color="#DD8452")
    axes[1, 0].hist(df["color_s_mean"].dropna(), bins=25, alpha=0.65, label="Độ bão hòa (S)", color="#4C72B0")
    axes[1, 0].set_title("Phân bố độ sáng (V) và độ bão hòa (S) trong vùng hạt")
    axes[1, 0].set_xlabel("Giá trị kênh (0 - 255)")
    axes[1, 0].set_ylabel("Số lượng hạt")
    axes[1, 0].legend()

    valid_groups = [g for g in quality_labels if (df["quality_label"] == g).sum() > 0]
    box_data = [df.loc[df["quality_label"] == g, "color_h_mean"].dropna().values for g in valid_groups]
    if box_data:
        axes[1, 1].boxplot(box_data, tick_labels=valid_groups)
        axes[1, 1].set_title("Phân bố sắc độ (H) theo nhóm chất lượng")
        axes[1, 1].set_ylabel("Hue (0 - 179)")
        axes[1, 1].tick_params(axis="x", rotation=20)

    plt.tight_layout()
    plt.show()


plot_tv4_statistics(df_classified, stats_report)